In [16]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import numpy as np
from evedesign.system import System, Protein
from evedesign.models.boltzfold import BoltzFoldTransformer

In [17]:
seq = "TSENPLLALREKISALDEKLLALLAERRELAVEVGKAKLLSHRPVRDIDRERDLLERLITLGKAHHLDAHYITRLFQLIIEDSVLTQQALLQQH"
s = System([Protein(rep=seq, id='EcCM', first_index=2)])
inst = s.rep_to_instance()

In [18]:
m = BoltzFoldTransformer(device='cpu', use_msa_server=True, diffusion_samples=5)
m.build(s)
predictions_dir, files = m.transform([inst])
cif_files = [f for f in files if f.suffix == ".cif"]
json_files = [f for f in files if f.suffix == ".json"]
npz_files = [f for f in files if f.suffix == ".npz"]
print(f"CIF: {[f.name for f in cif_files]}")
print(f"Confidence: {[f.name for f in json_files]}")
print(f"Arrays: {[f.name for f in npz_files]}")

Processing 1 inputs with 1 threads.


  0%|          | 0/1 [00:00<?, ?it/s]

Generating MSA for /var/folders/xb/vj5zk8_x0bs4xmp72n7_6dbm0000gn/T/boltzfold_svf9nraq/inputs/instance_0.yaml with 1 protein entities.
Calling MSA server for target instance_0 with 1 sequences
MSA server URL: https://api.colabfold.com
MSA pairing strategy: greedy
No authentication provided for MSA server


100%|██████████| 1/1 [00:04<00:00,  4.40s/it]
/Users/khbelahsen/Documents/GitHub/work/marks/evedesign/.venv/lib/python3.12/site-packages/pytorch_lightning/utilities/migration/utils.py:56: The loaded checkpoint was produced with Lightning v2.5.0.post0, which is newer than your current Lightning version: v2.5.0


KeyboardInterrupt: 

In [ ]:
import json
from evedesign.structure import StructureFile

if json_files:
    confidence = json.loads(json_files[0].read_text())
    print("Confidence scores:")
    for key, value in confidence.items():
        print(f"  {key}: {value}")

if cif_files:
    sf = StructureFile(str(cif_files[0]), format="cif")
    structure = sf.get_model()
    print(f"\nChains: {structure.chains()}")
    print(f"Atom count: {len(structure.atom_array)}")

/Users/khbelahsen/Documents/GitHub/work/marks/evedesign/.venv/lib/python3.12/site-packages/biotite/structure/io/pdbx/convert.py:461: UserWarning: Attribute 'auth_atom_id' not found within 'atom_site' category. The fallback attribute 'label_atom_id' will be used instead
  warnings.warn(
/Users/khbelahsen/Documents/GitHub/work/marks/evedesign/.venv/lib/python3.12/site-packages/biotite/structure/io/pdbx/convert.py:575: UserWarning: Missing 'pdbx_formal_charge' in 'atom_site' category. 'charge' will be set to 0
  warnings.warn(


Confidence scores:
  confidence_score: 0.9245090484619141
  ptm: 0.8647007942199707
  iptm: 0.0
  ligand_iptm: 0.0
  protein_iptm: 0.0
  complex_plddt: 0.9394611716270447
  complex_iplddt: 0.9394611716270447
  complex_pde: 0.34833136200904846
  complex_ipde: 0.0
  chains_ptm: {'0': 0.8647007942199707}
  pair_chains_iptm: {'0': {'0': 0.8647007942199707}}

Chains: ['A']
Atom count: 762


In [ ]:
! pip install py3Dmol -q

In [ ]:
import py3Dmol

if cif_files:
    view = py3Dmol.view(width=800, height=500)
    view.addModel(cif_files[0].read_text(), "cif")
    view.setStyle({
        "cartoon": {
            "colorscheme": {
                "prop": "b",
                "gradient": "roygb",
                "min": 50,
                "max": 90
            }
        }
    })
    view.zoomTo()
    view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### Multiple diffusion samples

In [ ]:
# Test: multiple diffusion samples
m_multi = BoltzFoldTransformer(
    device='cpu',
    use_msa_server=True,
    diffusion_samples=3,
)
m_multi.build(s)
predictions_dir_multi, files_multi = m_multi.transform([inst])

cif_multi = [f for f in files_multi if f.suffix == ".cif"]
json_multi = [f for f in files_multi if f.suffix == ".json"]
print(f"CIF files ({len(cif_multi)}):")
for f in cif_multi:
    print(f"  {f.name}")
print(f"JSON files ({len(json_multi)}):")
for f in json_multi:
    print(f"  {f.name}")

Processing 1 inputs with 1 threads.


  0%|          | 0/1 [00:00<?, ?it/s]

Generating MSA for /var/folders/xb/vj5zk8_x0bs4xmp72n7_6dbm0000gn/T/boltzfold_bs47eajg/inputs/instance_0.yaml with 1 protein entities.
Calling MSA server for target instance_0 with 1 sequences
MSA server URL: https://api.colabfold.com
MSA pairing strategy: greedy
No authentication provided for MSA server


100%|██████████| 1/1 [00:04<00:00,  4.42s/it]
/Users/khbelahsen/Documents/GitHub/work/marks/evedesign/.venv/lib/python3.12/site-packages/pytorch_lightning/utilities/migration/utils.py:56: The loaded checkpoint was produced with Lightning v2.5.0.post0, which is newer than your current Lightning version: v2.5.0
2026-04-10 12:49:57.288 | INFO     | evedesign.models.boltzfold:_load_model:205 - Boltz-2 loaded from /Users/khbelahsen/.boltz/boltz2_conf.ckpt
2026-04-10 13:00:24.980 | INFO     | evedesign.models.boltzfold:transform:360 - Boltz-2 output written to: /var/folders/xb/vj5zk8_x0bs4xmp72n7_6dbm0000gn/T/boltzfold_bs47eajg/predictions
2026-04-10 13:00:24.995 | INFO     | evedesign.models.boltzfold:transform:361 - Files written (16):
2026-04-10 13:00:24.998 | INFO     | evedesign.models.boltzfold:transform:364 -   instance_0/confidence_instance_0_model_0.json (442 bytes)
2026-04-10 13:00:24.999 | INFO     | evedesign.models.boltzfold:transform:364 -   instance_0/confidence_instance_0_mod

CIF files (3):
  instance_0_model_0.cif
  instance_0_model_1.cif
  instance_0_model_2.cif
JSON files (3):
  confidence_instance_0_model_0.json
  confidence_instance_0_model_1.json
  confidence_instance_0_model_2.json


In [ ]:
if cif_files:
    for cif_file in cif_multi:
        view = py3Dmol.view(width=200, height=200)
        view.addModel(cif_file.read_text(), "cif")
        view.setStyle({
            "cartoon": {
                "colorscheme": {
                    "prop": "b",
                    "gradient": "roygb",
                    "min": 50,
                    "max": 90
                }
            }
        })
        view.zoomTo()
        view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [21]:
import numpy as np
from evedesign.system import System, Protein, SystemInstance, EntityInstance
from evedesign.models.boltzfold import BoltzFoldTransformer

wt = "TSENPLLALREKISALDEKLLALLAERRELAVEVGKAKLLSHRPVRDIDRERDLLERLITLGKAHHLDAHYITRLFQLIIEDSVLTQQALLQQH"

# Bind the model to the WT system once
s = System([Protein(rep=wt, id='EcCM', first_index=2)])
m = BoltzFoldTransformer(device='cpu', use_msa_server=True).build(s)

# Generate all single-point mutants at, say, position 10 (first_index=2 → array idx 8)
AA = "ACDEFGHIKLMNPQRSTVWY"
variants = []
pos = 8  # array index in rep
for aa in AA:
    if aa == wt[pos]:
        continue
    mut = wt[:pos] + aa + wt[pos+1:]
    variants.append(mut)

# One SystemInstance per variant — all length 94, all match the bound system
instances = [
    SystemInstance([EntityInstance(rep=np.array(list(v), dtype='U1'))])
    for v in variants
]

predictions_dir, files = m.transform(instances)


Processing 19 inputs with 1 threads.


  0%|          | 0/19 [00:00<?, ?it/s]

Generating MSA for /var/folders/xb/vj5zk8_x0bs4xmp72n7_6dbm0000gn/T/boltzfold_sp_uf9w8/inputs/instance_0.yaml with 1 protein entities.
Calling MSA server for target instance_0 with 1 sequences
MSA server URL: https://api.colabfold.com
MSA pairing strategy: greedy
No authentication provided for MSA server


Sleeping for 6s. Reason: PENDING
Sleeping for 8s. Reason: RUNNING
  5%|▌         | 1/19 [00:19<05:56, 19.79s/it]

Generating MSA for /var/folders/xb/vj5zk8_x0bs4xmp72n7_6dbm0000gn/T/boltzfold_sp_uf9w8/inputs/instance_1.yaml with 1 protein entities.
Calling MSA server for target instance_1 with 1 sequences
MSA server URL: https://api.colabfold.com
MSA pairing strategy: greedy
No authentication provided for MSA server


...

Sleeping for 8s. Reason: PENDING
 89%|████████▉ | 17/19 [04:16<00:27, 13.82s/it]

Generating MSA for /var/folders/xb/vj5zk8_x0bs4xmp72n7_6dbm0000gn/T/boltzfold_sp_uf9w8/inputs/instance_17.yaml with 1 protein entities.
Calling MSA server for target instance_17 with 1 sequences
MSA server URL: https://api.colabfold.com
MSA pairing strategy: greedy
No authentication provided for MSA server


Sleeping for 8s. Reason: PENDING
 95%|█████████▍| 18/19 [04:29<00:13, 13.45s/it]

Generating MSA for /var/folders/xb/vj5zk8_x0bs4xmp72n7_6dbm0000gn/T/boltzfold_sp_uf9w8/inputs/instance_18.yaml with 1 protein entities.
Calling MSA server for target instance_18 with 1 sequences
MSA server URL: https://api.colabfold.com
MSA pairing strategy: greedy
No authentication provided for MSA server


Sleeping for 7s. Reason: PENDING
100%|██████████| 19/19 [04:41<00:00, 14.79s/it]
/Users/khbelahsen/Documents/GitHub/work/marks/evedesign/.venv/lib/python3.12/site-packages/pytorch_lightning/utilities/migration/utils.py:56: The loaded checkpoint was produced with Lightning v2.5.0.post0, which is newer than your current Lightning version: v2.5.0
2026-04-10 13:21:40.906 | INFO     | evedesign.models.boltzfold:_load_model:205 - Boltz-2 loaded from /Users/khbelahsen/.boltz/boltz2_conf.ckpt
2026-04-10 16:56:46.725 | INFO     | evedesign.models.boltzfold:transform:360 - Boltz-2 output written to: /var/folders/xb/vj5zk8_x0bs4xmp72n7_6dbm0000gn/T/boltzfold_sp_uf9w8/predictions
2026-04-10 16:56:46.759 | INFO     | evedesign.models.boltzfold:transform:361 - Files written (114):
2026-04-10 16:56:46.766 | INFO     | evedesign.models.boltzfold:transform:364 -   instance_0/confidence_instance_0_model_0.json (443 bytes)
2026-04-10 16:56:46.767 | INFO     | evedesign.models.boltzfold:transform:364 -   

In [22]:
import io
import biotite.structure as struc
import biotite.structure.io.pdbx as pdbx

# Pick a reference (e.g. the WT or the first variant) and load its CA atoms
ref_cif = pdbx.CIFFile.read(str(cif_files[0]))
ref_struct = pdbx.get_structure(ref_cif, model=1)
ref_ca = ref_struct[ref_struct.atom_name == "CA"]

view = py3Dmol.view(width=600, height=500)

# Reference structure: gray cartoon
view.addModel(cif_files[0].read_text(), "cif")

# Superimpose every other structure onto the reference and add aligned coords
for cif_file in cif_files[1:]:
    cif = pdbx.CIFFile.read(str(cif_file))
    mob = pdbx.get_structure(cif, model=1)
    mob_ca = mob[mob.atom_name == "CA"]

    # Returns the superimposed mobile structure + the transform
    fitted_ca, transform = struc.superimpose(ref_ca, mob_ca)
    fitted = transform.apply(mob)

    # Write the aligned structure to a CIF string and add to the view
    out = pdbx.CIFFile()
    pdbx.set_structure(out, fitted)
    buf = io.StringIO()
    out.write(buf)
    view.addModel(buf.getvalue(), "cif")

# Style: WT in gray, variants colored by pLDDT
view.setStyle({"model": 0}, {"cartoon": {"color": "gray"}})
view.setStyle({"model": -1}, {  # all subsequently added models
    "cartoon": {
        "colorscheme": {
            "prop": "b",
            "gradient": "roygb",
            "min": 50,
            "max": 90,
        }
    }
})
# (or, if "model": -1 doesn't apply to all, loop with explicit indices:)
for i in range(1, len(cif_files)):
    view.setStyle({"model": i}, {
        "cartoon": {
            "colorscheme": {
                "prop": "b",
                "gradient": "roygb",
                "min": 50,
                "max": 90,
            }
        }
    })

view.zoomTo()
view.show()

/Users/khbelahsen/Documents/GitHub/work/marks/evedesign/.venv/lib/python3.12/site-packages/biotite/structure/io/pdbx/convert.py:461: UserWarning: Attribute 'auth_atom_id' not found within 'atom_site' category. The fallback attribute 'label_atom_id' will be used instead
  warnings.warn(


3Dmol.js failed to load for some reason. Please check your browser console for error messages.